# PCA & FDA/LDA Analysis -- Irish Home BER Dataset

Full dimensionality-reduction pipeline:
1. Pre-PCA Feature Audit
2. Scaling
3. PCA Analysis
4. FDA / LDA Analysis
5. PCA vs LDA Comparison
6. Recommendations


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

DATA_FILE  = 'BER_Cleaned_Optimized.parquet'
TARGET_COL = 'BerRating'
FDA_LABEL  = 'DwellingTypeDescr'
DROP_FROM_FEATURES = ['CountyName', 'DwellingTypeDescr', 'BerRating']

print("Libraries loaded.")

Libraries loaded.


## Step 1 - Pre-PCA Feature Audit

Check categorical variance, flag near-zero-variance columns, decide encoding strategy.

In [2]:
print("Loading dataset ...")
df_raw = pd.read_parquet(DATA_FILE)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")
print(f"Missing values: {df_raw.isnull().sum().sum()}")

cat_cols   = df_raw.select_dtypes(include='category').columns.tolist()
float_cols = df_raw.select_dtypes(include=['float32','float64']).columns.tolist()

print(f"\n13 categorical columns: {cat_cols}")

NZV_THRESHOLD = 0.95
nzv_flags = []
print("\n--- Categorical Value Counts (top-1 share) ---")
for col in cat_cols:
    vc = df_raw[col].value_counts(normalize=True)
    top_share = vc.iloc[0]
    n_unique  = vc.shape[0]
    flag = "NZV FLAG" if top_share >= NZV_THRESHOLD else "OK"
    if top_share >= NZV_THRESHOLD:
        nzv_flags.append(col)
    print(f"  {col:<30s}  top={top_share:.1%}  unique={n_unique:>3d}  {flag}")

print(f"\nNear-zero-variance categoricals (>={NZV_THRESHOLD:.0%}) -> drop before PCA:")
for c in nzv_flags:
    print(f"   {c}")

print("\n--- Float near-zero-variance check ---")
float_nzv = []
for col in float_cols:
    if col in DROP_FROM_FEATURES:
        continue
    std = df_raw[col].std()
    if std < 1e-6:
        float_nzv.append(col)
        print(f"  {col} is near-constant (std={std:.2e}) -> DROP")
if not float_nzv:
    print("  All float columns have meaningful variance. OK")

print("\n--- Encoding Strategy ---")
yes_no_cols   = []
multicat_cols = []
ignore_cols   = DROP_FROM_FEATURES + nzv_flags

for col in cat_cols:
    if col in ignore_cols:
        continue
    uniq = set(str(v).upper() for v in df_raw[col].dropna().unique())
    if uniq <= {'YES', 'NO'}:
        yes_no_cols.append(col)
    else:
        multicat_cols.append(col)

print(f"  Binary (0/1):   {yes_no_cols}")
print(f"  One-hot:        {multicat_cols}")
print(f"  Dropped (geo/label/NZV): {ignore_cols}")

Loading dataset ...
Shape: 1,351,582 rows x 45 cols
Missing values: 0

13 categorical columns: ['CountyName', 'DwellingTypeDescr', 'VentilationMethod', 'StructureType', 'SuspendedWoodenFloor', 'CHBoilerThermostatControlled', 'OBBoilerThermostatControlled', 'OBPumpInsideDwelling', 'WarmAirHeatingSystem', 'UndergroundHeating', 'CylinderStat', 'CombinedCylinder', 'ThermalMassCategory']

--- Categorical Value Counts (top-1 share) ---
  CountyName                      top=9.6%  unique= 55  OK
  DwellingTypeDescr               top=30.2%  unique= 11  OK
  VentilationMethod               top=87.8%  unique=  6  OK
  StructureType                   top=88.8%  unique=  4  OK
  SuspendedWoodenFloor            top=92.6%  unique=  3  OK
  CHBoilerThermostatControlled    top=52.4%  unique=  2  OK
  OBBoilerThermostatControlled    top=90.4%  unique=  2  OK
  OBPumpInsideDwelling            top=93.0%  unique=  2  OK
  WarmAirHeatingSystem            top=99.7%  unique=  2  NZV FLAG
  UndergroundHeating 

## Step 2 - Encoding & Scaling

In [4]:
df = df_raw.copy()

# ── Separate continuous vs categorical ──
ignore_cols = DROP_FROM_FEATURES + nzv_flags  # CountyName, DwellingTypeDescr, BerRating + NZV cats

# Continuous features only (for PCA)
cont_cols = [c for c in float_cols if c not in ignore_cols]
print(f"Continuous features for PCA: {len(cont_cols)}")
print(f"  {cont_cols}")

# Categorical features (for FAMD/LDA – NOT for PCA)
cat_keep = [c for c in cat_cols if c not in ignore_cols]
print(f"\nCategorical features (excluded from PCA, used in FAMD/LDA): {len(cat_keep)}")
print(f"  {cat_keep}")

# ── Scale continuous features only ──
X = df[cont_cols].astype(np.float32).values
y_rating = df_raw[TARGET_COL].values
y_label  = df_raw[FDA_LABEL].astype(str).values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_cols = cont_cols  # PCA will use these

print(f"\nPCA feature matrix (continuous only): {X_scaled.shape}")
print(f"Feature count: {X_scaled.shape[1]}")


Continuous features for PCA: 31
  ['Year_of_Construction', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'NoStoreys', 'HSMainSystemEfficiency', 'HSEffAdjFactor', 'HSSupplHeatFraction', 'HSSupplSystemEff', 'WHMainSystemEff', 'WHEffAdjFactor', 'SupplSHFuel', 'SupplWHFuel', 'SHRenewableResources', 'WHRenewableResources', 'NoOfFansAndVents', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'HeatSystemControlCat', 'HeatSystemResponseCat', 'NoCentralHeatingPumps', 'DistributionLosses', 'ThermalBridgingFactor', 'LivingAreaPercent']

Categorical features (excluded from PCA, used in FAMD/LDA): 8
  ['VentilationMethod', 'StructureType', 'SuspendedWoodenFloor', 'CHBoilerThermostatControlled', 'OBBoilerThermostatControlled', 'OBPumpInsideDwelling', 'UndergroundHeating', 'ThermalMassCategory']

PCA feature matrix (continuous only): (1351582, 31)
Feature count: 31


## Step 3 - PCA Analysis

In [6]:
print("Running full PCA ...")
pca    = PCA()
X_pca  = pca.fit_transform(X_scaled)
ev_ratio = pca.explained_variance_ratio_
ev_cum   = np.cumsum(ev_ratio)

for threshold in [0.80, 0.90, 0.95]:
    n_comp = int(np.searchsorted(ev_cum, threshold)) + 1
    print(f"  Components for {threshold:.0%} variance: {n_comp}")

print("\n--- Cumulative Variance (All Components) ---")
print(f"{'PC':>4}  {'ExplVar%':>9}  {'CumVar%':>9}")
for i in range(len(ev_ratio)):
    print(f"  PC{i+1:>2d}  {ev_ratio[i]*100:>8.2f}%  {ev_cum[i]*100:>8.2f}%")

# --- Scree plot ---
n_show = min(40, len(ev_ratio))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, n_show+1), ev_ratio[:n_show]*100, color='steelblue', alpha=0.8)
axes[0].set(xlabel='Principal Component', ylabel='Explained Variance (%)', title='Scree Plot')
axes[1].plot(range(1, n_show+1), ev_cum[:n_show]*100, marker='o', markersize=3, color='steelblue')
axes[1].axhline(90, color='orange', linestyle='--', label='90%')
axes[1].axhline(95, color='red',    linestyle='--', label='95%')
axes[1].set(xlabel='Number of Components', ylabel='Cumulative Explained Variance (%)', title='Cumulative Variance')
axes[1].legend()
plt.tight_layout()
plt.savefig('pca_scree.png', dpi=150)
plt.close()
print("Saved: pca_scree.png")

# --- PC1 vs PC2 scatter ---
sample_idx = np.random.choice(len(X_pca), size=min(20000, len(X_pca)), replace=False)
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(X_pca[sample_idx, 0], X_pca[sample_idx, 1],
                c=y_rating[sample_idx], cmap='RdYlGn', alpha=0.3, s=3)
plt.colorbar(sc, ax=ax, label='BerRating')
ax.set(xlabel='PC1', ylabel='PC2', title='PC1 vs PC2 -- colored by BerRating (sample 20k)')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150)
plt.close()
print("Saved: pca_scatter.png")

# --- Biplot ---
loadings = pd.DataFrame(pca.components_.T, index=feature_cols,
                         columns=[f'PC{i+1}' for i in range(pca.n_components_)])
top10_pc1 = loadings['PC1'].abs().nlargest(10).index
scale_x = 1.0 / (X_pca[:, 0].max() - X_pca[:, 0].min())
scale_y = 1.0 / (X_pca[:, 1].max() - X_pca[:, 1].min())

fig, ax = plt.subplots(figsize=(11, 8))
ax.scatter(X_pca[sample_idx, 0]*scale_x, X_pca[sample_idx, 1]*scale_y,
           c=y_rating[sample_idx], cmap='RdYlGn', alpha=0.15, s=3)
for feat in top10_pc1:
    idx = feature_cols.index(feat)
    ax.annotate('', xy=(pca.components_[0, idx], pca.components_[1, idx]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='navy', lw=1.5))
    ax.text(pca.components_[0, idx]*1.05, pca.components_[1, idx]*1.05,
            feat, fontsize=7, color='navy')
ax.set(xlabel='PC1', ylabel='PC2', title='Biplot -- Top 10 PC1 Features')
plt.tight_layout()
plt.savefig('pca_biplot.png', dpi=150)
plt.close()
print("Saved: pca_biplot.png")

# --- Heatmap ---
top15_overall = loadings.iloc[:, :5].abs().max(axis=1).nlargest(15).index
heat_data = loadings.loc[top15_overall, ['PC1','PC2','PC3','PC4','PC5']]
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Top 15 Feature Loadings -- PC1 to PC5')
plt.tight_layout()
plt.savefig('pca_heatmap.png', dpi=150)
plt.close()
print("Saved: pca_heatmap.png")

# --- PC-BerRating correlation ---
print("\n--- Correlation of each PC with BerRating (PC1-PC10) ---")
for i in range(min(10, X_pca.shape[1])):
    corr = np.corrcoef(X_pca[:, i], y_rating)[0, 1]
    print(f"  PC{i+1:>2d}: r = {corr:+.4f}")

# --- Top 5 features per PC ---
print("\n--- Top 5 Contributing Features per PC ---")
for pc in ['PC1','PC2','PC3','PC4','PC5']:
    top5 = loadings[pc].abs().nlargest(5)
    print(f"  {pc}: {list(zip(top5.index, top5.values.round(3)))}")

Running full PCA ...
  Components for 80% variance: 14
  Components for 90% variance: 19
  Components for 95% variance: 23

--- Cumulative Variance (All Components) ---
  PC   ExplVar%    CumVar%
  PC 1     18.62%     18.62%
  PC 2     14.25%     32.88%
  PC 3      8.58%     41.46%
  PC 4      6.45%     47.91%
  PC 5      5.51%     53.42%
  PC 6      4.66%     58.08%
  PC 7      4.14%     62.22%
  PC 8      3.45%     65.66%
  PC 9      3.28%     68.95%
  PC10      3.18%     72.13%
  PC11      2.78%     74.90%
  PC12      2.64%     77.54%
  PC13      2.39%     79.94%
  PC14      2.37%     82.31%
  PC15      1.97%     84.28%
  PC16      1.83%     86.10%
  PC17      1.68%     87.78%
  PC18      1.60%     89.39%
  PC19      1.48%     90.87%
  PC20      1.38%     92.25%
  PC21      1.31%     93.56%
  PC22      1.23%     94.79%
  PC23      1.09%     95.88%
  PC24      1.04%     96.92%
  PC25      0.87%     97.80%
  PC26      0.76%     98.55%
  PC27      0.63%     99.18%
  PC28      0.38%    

## Step 4 - FDA / LDA Analysis

In [7]:
print("Running LDA ...")
le    = LabelEncoder()
y_enc = le.fit_transform(y_label)
n_classes         = len(le.classes_)
n_components_lda  = min(n_classes - 1, X_scaled.shape[1])

lda   = LDA(n_components=n_components_lda)
X_lda = lda.fit_transform(X_scaled, y_enc)

print(f"Classes: {list(le.classes_)}")
print(f"LDA components: {X_lda.shape[1]}")
print(f"Explained variance ratio: {lda.explained_variance_ratio_.round(3)}")

# --- LD1 vs LD2 scatter ---
palette = plt.cm.get_cmap('tab10', n_classes)
fig, ax = plt.subplots(figsize=(12, 8))
for i, cls in enumerate(le.classes_):
    mask = np.where(y_enc == i)[0]
    s_mask = np.random.choice(mask, size=min(3000, len(mask)), replace=False)
    ld2_vals = X_lda[s_mask, 1] if X_lda.shape[1] > 1 else np.zeros(len(s_mask))
    ax.scatter(X_lda[s_mask, 0], ld2_vals, color=palette(i), label=cls, alpha=0.4, s=5)
ax.set(xlabel='LD1', ylabel='LD2', title='LDA -- LD1 vs LD2 by DwellingTypeDescr (sample)')
ax.legend(fontsize=7, markerscale=3, loc='upper right')
plt.tight_layout()
plt.savefig('lda_scatter.png', dpi=150)
plt.close()
print("Saved: lda_scatter.png")

# --- Explained variance bar ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(lda.explained_variance_ratio_)+1),
       lda.explained_variance_ratio_*100, color='teal', alpha=0.8)
ax.set(xlabel='LDA Component', ylabel='Explained Variance (%)', title='LDA Explained Variance Ratio')
plt.tight_layout()
plt.savefig('lda_variance.png', dpi=150)
plt.close()
print("Saved: lda_variance.png")

# --- Top discriminating features ---
lda_coef_df = pd.DataFrame(lda.coef_, columns=feature_cols)
print("\n--- Top 5 discriminating features for LD1 ---")
ld1_top = lda_coef_df.iloc[0].abs().nlargest(5)
print(list(zip(ld1_top.index, ld1_top.values.round(3))))
if lda_coef_df.shape[0] > 1:
    print("--- Top 5 discriminating features for LD2 ---")
    ld2_top = lda_coef_df.iloc[1].abs().nlargest(5)
    print(list(zip(ld2_top.index, ld2_top.values.round(3))))

Running LDA ...
Classes: ['Apartment', 'Basement Dwelling', 'Detached house', 'End of terrace house', 'Ground-floor apartment', 'House', 'Maisonette', 'Mid-floor apartment', 'Mid-terrace house', 'Semi-detached house', 'Top-floor apartment']
LDA components: 10
Explained variance ratio: [0.526 0.26  0.169 0.024 0.011 0.006 0.003 0.001 0.001 0.   ]
Saved: lda_scatter.png
Saved: lda_variance.png

--- Top 5 discriminating features for LD1 ---
[('UValueFloor', np.float32(1.548)), ('FloorArea', np.float32(1.494)), ('HSSupplHeatFraction', np.float32(1.109)), ('LivingAreaPercent', np.float32(1.08)), ('HeatSystemResponseCat', np.float32(0.88))]
--- Top 5 discriminating features for LD2 ---
[('FloorArea', np.float32(7.663)), ('RoofArea', np.float32(7.538)), ('Year_of_Construction', np.float32(1.705)), ('LivingAreaPercent', np.float32(1.593)), ('WallArea', np.float32(1.495))]


## Step 5 - PCA vs LDA vs Original: Classification Accuracy (5-fold CV)

In [8]:
print("Running 5-fold CV comparison (sub-sampled to 100k rows) ...")

rng    = np.random.default_rng(42)
idx_cv = rng.choice(len(X_scaled), size=min(100_000, len(X_scaled)), replace=False)
X_cv   = X_scaled[idx_cv]
y_cv   = y_enc[idx_cv]

n90      = int(np.searchsorted(ev_cum, 0.90)) + 1
X_pca90  = X_pca[idx_cv, :n90]
X_lda_cv = X_lda[idx_cv]

CV      = 5
results = {}
label_map = {
    'Original':               X_cv,
    f'PCA (n={n90})':         X_pca90,
    f'LDA (n={X_lda_cv.shape[1]})': X_lda_cv,
}

for label, Xf in label_map.items():
    lr_scores = cross_val_score(
        LogisticRegression(max_iter=500, random_state=42),
        Xf, y_cv, cv=CV, scoring='accuracy', n_jobs=-1)
    rf_scores = cross_val_score(
        RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        Xf, y_cv, cv=CV, scoring='accuracy')
    results[label] = {
        'LogReg Acc': f"{lr_scores.mean():.4f} +/- {lr_scores.std():.4f}",
        'RF Acc':     f"{rf_scores.mean():.4f} +/- {rf_scores.std():.4f}"
    }
    print(f"  {label}: LogReg={lr_scores.mean():.4f}, RF={rf_scores.mean():.4f}")

print("\n=== COMPARISON TABLE ===")
df_results = pd.DataFrame(results).T
print(df_results.to_string())

Running 5-fold CV comparison (sub-sampled to 100k rows) ...
  Original: LogReg=0.7799, RF=0.7903
  PCA (n=19): LogReg=0.6637, RF=0.6568
  LDA (n=10): LogReg=0.7631, RF=0.7862

=== COMPARISON TABLE ===
                   LogReg Acc             RF Acc
Original    0.7799 +/- 0.0008  0.7903 +/- 0.0020
PCA (n=19)  0.6637 +/- 0.0017  0.6568 +/- 0.0025
LDA (n=10)  0.7631 +/- 0.0012  0.7862 +/- 0.0017


## Step 6 - Recommendation and MCA (Multiple Correspondence Analysis) — Categorical Features 

In [9]:
n80 = int(np.searchsorted(ev_cum, 0.80)) + 1
n90 = int(np.searchsorted(ev_cum, 0.90)) + 1
n95 = int(np.searchsorted(ev_cum, 0.95)) + 1

print("=== FINAL RECOMMENDATIONS ===")
print()
print("FEATURE ENGINEERING:")
print(f"  Drop near-zero-variance categoricals: {nzv_flags}")
print(f"  Drop float NZV columns:               {float_nzv if float_nzv else 'None found'}")
print("  Drop CountyName (geographic identifier, not building feature)")
print()
print("DIMENSIONALITY REDUCTION:")
print(f"  PCA:  {n80} components -> 80% variance")
print(f"  PCA:  {n90} components -> 90% variance  <-- RECOMMENDED for regression")
print(f"  PCA:  {n95} components -> 95% variance")
print(f"  LDA:  {n_components_lda} components (class-driven, best for classification)")
print()
print("DOWNSTREAM MODELLING:")
print(f"  BerRating regression     -> use PCA n={n90} features")
print("  DwellingType classification -> use LDA features (better cluster separation)")
print("  Random Forest baseline   -> original features viable (tree-based, scale-free)")
print()
print("PLOTS SAVED:")
print("  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png")
print("  lda_scatter.png, lda_variance.png")

=== FINAL RECOMMENDATIONS ===

FEATURE ENGINEERING:
  Drop near-zero-variance categoricals: ['WarmAirHeatingSystem', 'CylinderStat', 'CombinedCylinder']
  Drop float NZV columns:               None found
  Drop CountyName (geographic identifier, not building feature)

DIMENSIONALITY REDUCTION:
  PCA:  14 components -> 80% variance
  PCA:  19 components -> 90% variance  <-- RECOMMENDED for regression
  PCA:  23 components -> 95% variance
  LDA:  10 components (class-driven, best for classification)

DOWNSTREAM MODELLING:
  BerRating regression     -> use PCA n=19 features
  DwellingType classification -> use LDA features (better cluster separation)
  Random Forest baseline   -> original features viable (tree-based, scale-free)

PLOTS SAVED:
  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png
  lda_scatter.png, lda_variance.png


In [ ]:
## ── Step X: MCA (Multiple Correspondence Analysis) — Categorical Features ──

import prince

# Select categorical features (exclude NZV + geo + target)
mca_drop = set(nzv_flags + ['CountyName'])  # already identified for dropping
mca_cols = [c for c in cat_cols if c not in mca_drop]
print(f"MCA input: {len(mca_cols)} categorical features")
print(f"  {mca_cols}")

# Prepare categorical DataFrame (string type required by prince)
df_mca = df_raw[mca_cols].astype(str)

# Subsample for speed (MCA on 1.35M rows is slow)
N_MCA = 50_000
rng_mca = np.random.default_rng(42)
idx_mca = rng_mca.choice(len(df_mca), size=min(N_MCA, len(df_mca)), replace=False)
df_mca_sample = df_mca.iloc[idx_mca].reset_index(drop=True)

# Fit MCA — n_components = total categories - n_columns (max possible)
n_components_mca = min(20, df_mca_sample.nunique().sum() - len(mca_cols))
mca = prince.MCA(n_components=n_components_mca, random_state=42)
mca = mca.fit(df_mca_sample)

# ── Explained Inertia ──
inertia = mca.percentage_of_variance_  # per-component inertia %
cum_inertia = np.cumsum(inertia)

print(f"\n--- MCA Explained Inertia (Top {n_components_mca} Components) ---")
print(f"  {'Comp':<6} {'Inertia%':>10} {'Cum%':>10}")
for i in range(n_components_mca):
    print(f"  F{i+1:<4d} {inertia[i]:>9.2f}% {cum_inertia[i]:>9.2f}%")

for t in [80, 90, 95]:
    idx_t = np.searchsorted(cum_inertia, t)
    if idx_t < len(cum_inertia):
        print(f"  Components for {t}% inertia: {idx_t + 1}")
    else:
        print(f"  Components for {t}% inertia: >{n_components_mca} (need more components)")

# ── Column Contributions ──
col_contribs = mca.column_contributions_
print("\n--- Top Contributing Categories per MCA Dimension ---")
for dim in range(min(3, n_components_mca)):
    top5 = col_contribs[dim].nlargest(5)
    print(f"  Dim {dim+1}: {list(zip(top5.index, top5.values.round(4)))}")

# ── Scree Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_plot = len(inertia)
axes[0].bar(range(1, n_plot+1), inertia, color='mediumorchid', alpha=0.8)
axes[0].set(xlabel='MCA Component', ylabel='Inertia (%)', title='MCA Scree Plot (Categorical Features)')
axes[1].plot(range(1, n_plot+1), cum_inertia, marker='o', markersize=4, color='mediumorchid')
axes[1].axhline(80, color='orange', linestyle='--', label='80%')
axes[1].axhline(90, color='red', linestyle='--', label='90%')
axes[1].set(xlabel='Number of Components', ylabel='Cumulative Inertia (%)', title='MCA Cumulative Inertia')
axes[1].legend()
plt.tight_layout()
plt.savefig('mca_scree.png', dpi=150)
plt.close()
print("\nSaved: mca_scree.png")

# ── Factor Map (Dim1 vs Dim2) colored by DwellingTypeDescr ──
row_coords = mca.row_coordinates(df_mca_sample)
y_mca = df_mca_sample['DwellingTypeDescr'].values if 'DwellingTypeDescr' in mca_cols else None

fig, ax = plt.subplots(figsize=(11, 8))
if y_mca is not None:
    le_mca = LabelEncoder()
    y_mca_enc = le_mca.fit_transform(y_mca)
    sc = ax.scatter(row_coords[0], row_coords[1],
                    c=y_mca_enc, cmap='tab10', alpha=0.3, s=3)
    handles = [plt.Line2D([0],[0], marker='o', linestyle='', color=plt.cm.tab10(i/len(le_mca.classes_)),
               label=c, markersize=5) for i, c in enumerate(le_mca.classes_)]
    ax.legend(handles=handles, fontsize=7, loc='upper right', markerscale=2)
else:
    ax.scatter(row_coords[0], row_coords[1], alpha=0.3, s=3, color='mediumorchid')
ax.set(xlabel='MCA Dim 1', ylabel='MCA Dim 2',
       title='MCA — Dim1 vs Dim2 (Categorical Features, colored by DwellingType)')
plt.tight_layout()
plt.savefig('mca_scatter.png', dpi=150)
plt.close()
print("Saved: mca_scatter.png")
# ── Save MCA output to file ──
mca_buffer = []
mca_buffer.append("\n" + "=" * 60)
mca_buffer.append("MCA ANALYSIS (Categorical Features Only)")
mca_buffer.append("=" * 60)
mca_buffer.append(f"MCA input: {len(mca_cols)} categorical features")
mca_buffer.append(f"  {mca_cols}")
mca_buffer.append(f"Sample size: {len(df_mca_sample):,} rows")
mca_buffer.append(f"Components fitted: {n_components_mca}")

mca_buffer.append(f"\n--- MCA Explained Inertia ---")
mca_buffer.append(f"  {'Comp':<6} {'Inertia%':>10} {'Cum%':>10}")
for i in range(n_components_mca):
    mca_buffer.append(f"  F{i+1:<4d} {inertia[i]:>9.2f}% {cum_inertia[i]:>9.2f}%")

for t in [80, 90, 95]:
    idx_t = np.searchsorted(cum_inertia, t)
    if idx_t < len(cum_inertia):
        mca_buffer.append(f"  Components for {t}% inertia: {idx_t + 1}")
    else:
        mca_buffer.append(f"  Components for {t}% inertia: >{n_components_mca}")

mca_buffer.append(f"\n--- Top Contributing Categories per MCA Dimension ---")
for dim in range(min(3, n_components_mca)):
    top5 = col_contribs[dim].nlargest(5)
    mca_buffer.append(f"  Dim {dim+1}:")
    for cat, val in zip(top5.index, top5.values):
        mca_buffer.append(f"    {cat:<50s} {val:.4f}")

mca_buffer.append(f"\nPlots saved: mca_scree.png, mca_scatter.png")

# Append to pca_summary_output.txt
with open('pca_summary_output.txt', 'a', encoding='utf-8') as f:
    f.write('\n'.join(mca_buffer) + '\n')

print("\nMCA output appended to: pca_summary_output.txt")



MCA input: 9 categorical features
  ['DwellingTypeDescr', 'VentilationMethod', 'StructureType', 'SuspendedWoodenFloor', 'CHBoilerThermostatControlled', 'OBBoilerThermostatControlled', 'OBPumpInsideDwelling', 'UndergroundHeating', 'ThermalMassCategory']

--- MCA Explained Inertia (Top 20 Components) ---
  Comp     Inertia%       Cum%
  F1         6.83%      6.83%
  F2         5.57%     12.39%
  F3         5.09%     17.49%
  F4         4.55%     22.04%
  F5         4.05%     26.09%
  F6         3.88%     29.97%
  F7         3.72%     33.69%
  F8         3.66%     37.35%
  F9         3.64%     40.99%
  F10        3.61%     44.59%
  F11        3.59%     48.18%
  F12        3.58%     51.76%
  F13        3.57%     55.34%
  F14        3.57%     58.91%
  F15        3.54%     62.45%
  F16        3.52%     65.97%
  F17        3.52%     69.49%
  F18        3.47%     72.96%
  F19        3.45%     76.42%
  F20        3.41%     79.83%
  Components for 80% inertia: >20 (need more components)
  Compon

Exception ignored in: <function ResourceTracker.__del__ at 0x104305da0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107f2dda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107ca1da0>
Traceback (most recent call last

In [16]:
import io
buffer = io.StringIO()
buffer.write("=== FINAL RECOMMENDATIONS ===")
buffer.write("\n")
buffer.write("FEATURE ENGINEERING:\n")
buffer.write(f"  Drop near-zero-variance categoricals: {nzv_flags}\n")
buffer.write(f"  Drop float NZV columns:               {float_nzv if float_nzv else 'None found'}\n")
buffer.write("  Drop CountyName (geographic identifier, not building feature)\n")
buffer.write("\n")
buffer.write("DIMENSIONALITY REDUCTION:\n")
buffer.write(f"  PCA:  {n80} components -> 80% variance\n")
buffer.write(f"  PCA:  {n90} components -> 90% variance  <-- RECOMMENDED for regression\n")
buffer.write(f"  PCA:  {n95} components -> 95% variance\n")
buffer.write(f"  LDA:  {n_components_lda} components (class-driven, best for classification)\n")
buffer.write("\n")
buffer.write("DOWNSTREAM MODELLING:\n")
buffer.write(f"  BerRating regression     -> use PCA n={n90} features\n")
buffer.write("  DwellingType classification -> use LDA features (better cluster separation)\n")
buffer.write("  Random Forest baseline   -> original features viable (tree-based, scale-free)\n")
buffer.write("\n")
buffer.write("PLOTS SAVED:\n")
buffer.write("  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png\n")
buffer.write("  lda_scatter.png, lda_variance.png\n")
info_buf = io.StringIO()
buffer.write(info_buf.getvalue())

summary_content = buffer.getvalue()
print(summary_content)
# Save to file
summary_file = 'pca_summary_output.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write(summary_content)

print(f"\nValidation summary saved to: {summary_file}")

=== FINAL RECOMMENDATIONS ===
FEATURE ENGINEERING:
  Drop near-zero-variance categoricals: ['WarmAirHeatingSystem', 'CylinderStat', 'CombinedCylinder']
  Drop float NZV columns:               None found
  Drop CountyName (geographic identifier, not building feature)

DIMENSIONALITY REDUCTION:
  PCA:  14 components -> 80% variance
  PCA:  19 components -> 90% variance  <-- RECOMMENDED for regression
  PCA:  23 components -> 95% variance
  LDA:  10 components (class-driven, best for classification)

DOWNSTREAM MODELLING:
  BerRating regression     -> use PCA n=19 features
  DwellingType classification -> use LDA features (better cluster separation)
  Random Forest baseline   -> original features viable (tree-based, scale-free)

PLOTS SAVED:
  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png
  lda_scatter.png, lda_variance.png


Validation summary saved to: pca_summary_output.txt


## Step 7 - FAMD (Factor Analysis of Mixed Data)

FAMD natively handles mixed float + categorical columns — no manual encoding required.
It extends PCA to work jointly on continuous and categorical features, decomposing variance across both types simultaneously.
Predicting target: `BerRating` (continuous energy rating).

In [14]:
# Auto-install prince if needed
import subprocess, sys, io
subprocess.check_call([sys.executable, "-m", "pip", "install", "prince", "--quiet"])
import prince

famd_buf = io.StringIO()
def fprint(text=""):
    print(text)
    famd_buf.write(str(text) + "\n")

fprint("Running FAMD on raw mixed-type data ...")

# Build the FAMD feature DataFrame
FAMD_DROP = ['BerRating', 'CountyName']
df_famd = df_raw.drop(columns=[c for c in FAMD_DROP if c in df_raw.columns], errors='ignore').copy()

for col in df_famd.select_dtypes(include='category').columns:
    df_famd[col] = df_famd[col].astype(str).astype('category')

# Sample for speed
rng2 = np.random.default_rng(0)
famd_idx  = rng2.choice(len(df_famd), size=min(50_000, len(df_famd)), replace=False)
df_famd_s = df_famd.iloc[famd_idx].reset_index(drop=True)
y_famd_s  = y_rating[famd_idx]

fprint(f"FAMD input shape: {df_famd_s.shape}")

N_FAMD = min(30, df_famd_s.shape[1])
famd   = prince.FAMD(n_components=N_FAMD, n_iter=3, random_state=42)
famd   = famd.fit(df_famd_s)

# -- Version-agnostic Explained Variance --
famd_ev = None
try:
    if hasattr(famd, 'eigenvalues_summary'):
        ev_summary = famd.eigenvalues_summary
        pct_col = [c for c in ev_summary.columns if 'variance' in c.lower() and '%' in c][0]
        vals = ev_summary[pct_col].values
        if isinstance(vals[0], str):
            famd_ev = np.array([float(x.strip('%')) / 100.0 for x in vals])
        else:
            famd_ev = vals / 100.0
    elif hasattr(famd, 'explained_inertia_'):
        famd_ev = np.array(famd.explained_inertia_)
    elif hasattr(famd, 'eigenvalues_'):
        eigs = np.array(famd.eigenvalues_)
        famd_ev = eigs / eigs.sum()
except Exception as e:
    fprint(f"Warning calculating variance: {e}")
    famd_ev = np.zeros(N_FAMD)

if famd_ev is None or len(famd_ev) == 0:
    famd_ev = np.zeros(N_FAMD)

famd_ev_cum = np.cumsum(famd_ev)

# -- Column coordinates (extracted early to find drivers) --
col_coords = None
try:
    if hasattr(famd, 'column_coordinates'):
        try:
            col_coords = famd.column_coordinates(df_famd_s)
        except:
            col_coords = famd.column_coordinates(df_famd_s, X=df_famd_s)
    elif hasattr(famd, 'column_coordinates_'):
        col_coords = famd.column_coordinates_
    elif hasattr(famd, 'column_contributions_'):
        col_coords = famd.column_contributions_
except Exception as e:
    fprint(f"Warning getting column coordinates: {e}")

fprint("\n--- FAMD Explained Inertia (All Components) ---")
fprint(f"{'Comp':>6}  {'Top Driver Feature':<30}  {'Inertia%':>10}  {'Cum%':>8}")
for i in range(len(famd_ev)):
    driver = "Unknown"
    if col_coords is not None and not col_coords.empty and i < col_coords.shape[1]:
        driver = col_coords.iloc[:, i].abs().idxmax()
        # Trim the driver string if it's too long
        if len(driver) > 30:
            driver = driver[:27] + "..."
    fprint(f"  F{i+1:>2d}   {str(driver):<30}  {famd_ev[i]*100:>9.2f}%  {famd_ev_cum[i]*100:>7.2f}%")

for thr in [0.80, 0.90, 0.95]:
    idx = int(np.searchsorted(famd_ev_cum, thr))
    n_f = min(idx + 1, len(famd_ev))
    fprint(f"  Components for {thr:.0%} inertia: {n_f}")

# -- Row coordinates for plot --
try:
    if hasattr(famd, 'row_coordinates'):
        row_coords = famd.row_coordinates(df_famd_s)
    elif hasattr(famd, 'row_coordinates_'):
        row_coords = famd.row_coordinates_
    else:
        row_coords = pd.DataFrame(np.zeros((len(df_famd_s), 2)))
except Exception as e:
    fprint(f"Warning getting row coordinates: {e}")
    row_coords = pd.DataFrame(np.zeros((len(df_famd_s), 2)))

# -- Scree / cumulative plots --
n_show = min(20, len(famd_ev))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, n_show+1), famd_ev[:n_show]*100, color='darkorange', alpha=0.8)
axes[0].set(xlabel='FAMD Component', ylabel='Explained Inertia (%)', title='FAMD Scree Plot')
axes[1].plot(range(1, n_show+1), famd_ev_cum[:n_show]*100, marker='o', markersize=4, color='darkorange')
axes[1].axhline(90, color='dodgerblue', linestyle='--', label='90%')
axes[1].axhline(95, color='red',        linestyle='--', label='95%')
axes[1].set(xlabel='Number of FAMD Components', ylabel='Cumulative Inertia (%)', title='FAMD Cumulative Inertia')
axes[1].legend()
plt.tight_layout()
plt.savefig('famd_variance.png', dpi=150)
plt.close()
fprint("NOTE: This FAMD includes DwellingTypeDescr and low-variance columns.")
fprint("Saved: famd_variance.png")

# -- Factor 1 vs Factor 2 scatter --
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(row_coords.iloc[:, 0], row_coords.iloc[:, 1],
                c=y_famd_s, cmap='RdYlGn', alpha=0.3, s=4)
plt.colorbar(sc, ax=ax, label='BerRating')
ax.set(xlabel='FAMD Factor 1', ylabel='FAMD Factor 2',
       title='FAMD Factor 1 vs Factor 2 -- colored by BerRating (50k sample)')
plt.tight_layout()
plt.savefig('famd_scatter.png', dpi=150)
plt.close()
fprint("Saved: famd_scatter.png")

if col_coords is not None and not col_coords.empty:
    fprint("\n--- Top 10 columns contributing to FAMD Factor 1 ---")
    f1_top = col_coords.iloc[:, 0].abs().nlargest(10)
    fprint(f1_top.round(4).to_string())
    if col_coords.shape[1] > 1:
        fprint("\n--- Top 10 columns contributing to FAMD Factor 2 ---")
        f2_top = col_coords.iloc[:, 1].abs().nlargest(10)
        fprint(f2_top.round(4).to_string())
else:
    fprint("\nCould not retrieve column coordinates for this prince version.")

summary_file = 'pca_summary_output.txt'
with open(summary_file, 'a', encoding='utf-8') as f:
    f.write("\n" + "="*60 + "\n")
    f.write("STEP 7: FAMD ANALYSIS\n")
    f.write("="*60 + "\n")
    f.write(famd_buf.getvalue())
print(f"\nAppended Step 7 report to {summary_file}")


Running FAMD on raw mixed-type data ...
FAMD input shape: (50000, 43)

--- FAMD Explained Inertia (All Components) ---
  Comp  Top Driver Feature                Inertia%      Cum%
  F 1   VentilationMethod                    6.95%     6.95%
  F 2   DwellingTypeDescr                    6.26%    13.21%
  F 3   CylinderStat                         4.46%    17.67%
  F 4   DwellingTypeDescr                    4.30%    21.97%
  F 5   StructureType                        3.70%    25.67%
  F 6   OBBoilerThermostatControlled         3.23%    28.90%
  F 7   DwellingTypeDescr                    3.14%    32.04%
  F 8   DwellingTypeDescr                    2.98%    35.02%
  F 9   DwellingTypeDescr                    2.94%    37.96%
  F10   DwellingTypeDescr                    2.87%    40.83%
  F11   DwellingTypeDescr                    2.84%    43.67%
  F12   DwellingTypeDescr                    2.82%    46.49%
  F13   DwellingTypeDescr                    2.81%    49.30%
  F14   DwellingTypeDescr  

Exception ignored in: <function ResourceTracker.__del__ at 0x102d3dda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x11103dda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1088d1da0>
Traceback (most recent call last

## Step 8 - PCA Feature Column Report

Which original feature columns matter most for the 31-PC (90% variance) and 37-PC (95% variance) BerRating regression models?

This ranks every feature by its aggregate absolute loading across the selected PCs.

In [17]:
# PCA loadings are already in `loadings` (computed in Step 3)
# loadings index = feature_cols, columns = PC1..PCn
import io

n_90 = int(np.searchsorted(ev_cum, 0.90)) + 1
n_95 = int(np.searchsorted(ev_cum, 0.95)) + 1

rep_buf = io.StringIO()
def bprint(text=""):
    print(text)
    rep_buf.write(str(text) + "\n")

bprint(f"PCA 90% threshold -> {n_90} components")
bprint(f"PCA 95% threshold -> {n_95} components")

def feature_importance_report(loadings_df, n_pcs, threshold_label):
    pc_cols = [f'PC{i+1}' for i in range(n_pcs)]
    # Aggregate importance = sum of absolute loadings across the selected PCs
    agg_importance = loadings_df[pc_cols].abs().sum(axis=1).sort_values(ascending=False)
    
    report_lines = []
    report_lines.append(f"=== PCA Feature Column Report ({threshold_label}, {n_pcs} PCs) ===")
    report_lines.append(f"{'Rank':>5}  {'Aggregate Loading':>18}  Feature")
    report_lines.append("-" * 60)
    for rank, (feat, score) in enumerate(agg_importance.items(), 1):
        report_lines.append(f"  {rank:>3d}  {score:>18.4f}  {feat}")
    
    # Bottom 10: candidates to drop
    report_lines.append("")
    report_lines.append("--- Bottom 10 (lowest contribution -- candidates to drop) ---")
    bottom10 = agg_importance.tail(10)
    for feat, score in bottom10.items():
        report_lines.append(f"  {feat:<45s}  {score:.4f}")
    
    return "\n".join(report_lines), agg_importance

# --- 90% report ---
report_90, imp_90 = feature_importance_report(loadings, n_90, "90% variance")
bprint("\n" + report_90)

with open('features_pca_90pct.txt', 'w') as f:
    f.write(report_90)
bprint(f"\nSaved: {'features_pca_90pct.txt'}")

# --- 95% report ---
report_95, imp_95 = feature_importance_report(loadings, n_95, "95% variance")
bprint("\n" + report_95)

with open('features_pca_95pct.txt', 'w') as f:
    f.write(report_95)
bprint(f"Saved: {'features_pca_95pct.txt'}")

# --- Overlap: features in BOTH top-31 and top-37 ----
top31_set = set(imp_90.head(n_90).index)
top37_set = set(imp_95.head(n_95).index)
drop_candidates = set(imp_90.tail(10).index)

bprint("\n=== SUMMARY FOR BerRating REGRESSION ===")
bprint(f"  Features strongly driving 90% model ({n_90} PCs): {len(top31_set)}")
bprint(f"  Features strongly driving 95% model ({n_95} PCs): {len(top37_set)}")
bprint("\n  Top 15 most important features (90% model):")
for i, (f, v) in enumerate(imp_90.head(15).items(), 1):
    bprint(f"    {i:>2d}. {f:<45s}  loading={v:.4f}")

bprint("\n  Lowest-impact features (drop candidates for lean model):")
for f in drop_candidates:
   bprint(f"     {f}")

summary_file = 'pca_summary_output.txt'
with open(summary_file, 'a', encoding='utf-8') as f:
    f.write("\n" + "="*60 + "\n")
    f.write("STEP 8: PCA FEATURE IMPORTANCE REPORT\n")
    f.write("="*60 + "\n")
    f.write(rep_buf.getvalue())
print(f"\nAppended Step 8 report to {summary_file}")


PCA 90% threshold -> 19 components
PCA 95% threshold -> 23 components

=== PCA Feature Column Report (90% variance, 19 PCs) ===
 Rank   Aggregate Loading  Feature
------------------------------------------------------------
    1              3.5555  PercentageDraughtStripped
    2              3.4726  HeatSystemResponseCat
    3              3.2888  SupplWHFuel
    4              3.2236  NoOfFansAndVents
    5              3.1430  DoorArea
    6              3.0078  NoCentralHeatingPumps
    7              2.9987  ThermalBridgingFactor
    8              2.9703  NoOfSidesSheltered
    9              2.7076  NoStoreys
   10              2.6980  UvalueDoor
   11              2.6293  HSSupplSystemEff
   12              2.5744  LivingAreaPercent
   13              2.5725  UValueRoof
   14              2.5433  HSMainSystemEfficiency
   15              2.4238  WHMainSystemEff
   16              2.4209  Year_of_Construction
   17              2.4155  HeatSystemControlCat
   18              2

---
## Step 9 — Multicollinearity Audit (Continuous Features)

Calculate VIF for all continuous/float columns (excluding `BerRating`), flag features with VIF > 5 and > 10, and generate a correlation heatmap for the top 20 most inter-correlated continuous features.

In [18]:
# ── Step 9: VIF for continuous features ─────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

print("=" * 60)
print("STEP 9: MULTICOLLINEARITY AUDIT — CONTINUOUS FEATURES")
print("=" * 60)

# Load data (reuse if already in memory, else reload)
try:
    _shape = df_raw.shape
except NameError:
    df_raw = pd.read_parquet('BER_Cleaned_Optimized.parquet')

# Identify continuous columns (float), exclude target
float_cols = df_raw.select_dtypes(include=['float32', 'float64']).columns.tolist()
cont_cols  = [c for c in float_cols if c != 'BerRating']
print(f"Continuous feature columns ({len(cont_cols)}): {cont_cols}")

# Drop rows with any NaN in these columns for VIF calculation
X_cont = df_raw[cont_cols].dropna().astype(float)

# VIF can be slow on 1.35M rows — sample 100k
rng = np.random.default_rng(42)
idx = rng.choice(len(X_cont), size=min(100_000, len(X_cont)), replace=False)
X_vif = X_cont.iloc[idx].reset_index(drop=True)

print(f"\nCalculating VIF on {len(X_vif):,} rows ...")
X_const = add_constant(X_vif)
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_const.values, i + 1)
            for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print("\n--- VIF Scores (Ranked High → Low) ---")
print(f"{'Rank':<5} {'Feature':<35} {'VIF':>10}  Flag")
print("-" * 60)
for _, row in vif_data.iterrows():
    flag = ""
    if row['VIF'] > 10:
        flag = "⚠️  SEVERE (>10)"
    elif row['VIF'] > 5:
        flag = "⚡ MODERATE (>5)"
    print(f"{_+1:<5} {row['Feature']:<35} {row['VIF']:>10.2f}  {flag}")

severe   = vif_data[vif_data['VIF'] > 10]
moderate = vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] <= 10)]
print(f"\nSevere (VIF>10):   {len(severe)} features → {severe['Feature'].tolist()}")
print(f"Moderate (VIF>5):  {len(moderate)} features → {moderate['Feature'].tolist()}")

# ── Correlation heatmap: top 20 most correlated pairs ────────────────────────
corr = X_vif.corr().abs()
# Sum of abs correlations per feature (excluding self)
corr_sum = (corr.sum() - 1).sort_values(ascending=False)
top20    = corr_sum.head(20).index.tolist()
corr_top = X_vif[top20].corr()

# Flag pairs above 0.7
high_pairs = []
for i in range(len(top20)):
    for j in range(i+1, len(top20)):
        v = corr_top.iloc[i, j]
        if abs(v) > 0.7:
            high_pairs.append((top20[i], top20[j], round(v, 3)))

print(f"\nHighly correlated pairs (|r|>0.7) among top-20 continuous features:")
for a, b, v in sorted(high_pairs, key=lambda x: -abs(x[2])):
    print(f"  {a:30s} <-> {b:30s}  r={v}")

mask = np.triu(np.ones_like(corr_top, dtype=bool))
annot_data = corr_top.copy()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_top, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.4, ax=ax,
            annot_kws={'size': 7})
# Overlay strong correlations in bold
for i in range(len(top20)):
    for j in range(len(top20)):
        if not mask[i, j] and abs(corr_top.iloc[i, j]) > 0.7:
            ax.texts[i * len(top20) + j - sum(mask[k, :].sum() for k in range(i)) - mask[i, :j+1].sum() + 1 - 1]

ax.set_title('Correlation Heatmap — Top 20 Continuous Features (lower triangle)', fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('vif_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: vif_correlation_heatmap.png")


STEP 9: MULTICOLLINEARITY AUDIT — CONTINUOUS FEATURES
Continuous feature columns (31): ['Year_of_Construction', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'NoStoreys', 'HSMainSystemEfficiency', 'HSEffAdjFactor', 'HSSupplHeatFraction', 'HSSupplSystemEff', 'WHMainSystemEff', 'WHEffAdjFactor', 'SupplSHFuel', 'SupplWHFuel', 'SHRenewableResources', 'WHRenewableResources', 'NoOfFansAndVents', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'HeatSystemControlCat', 'HeatSystemResponseCat', 'NoCentralHeatingPumps', 'DistributionLosses', 'ThermalBridgingFactor', 'LivingAreaPercent']

Calculating VIF on 100,000 rows ...

--- VIF Scores (Ranked High → Low) ---
Rank  Feature                                    VIF  Flag
------------------------------------------------------------
1     WHRenewableResources                     77.94  ⚠️  SEVERE (>10)
2     SHRenewableResources                     77.90  ⚠️

---
## Step 10 — Multicollinearity Audit (Categorical Features)

Calculate Cramér's V for all pairs of the 12 categorical columns and generate an annotated heatmap. Flag pairs with Cramér's V > 0.5 as strongly associated.

In [19]:
# ── Step 10: Cramér's V for categorical features ─────────────────────────────
from scipy.stats import chi2_contingency

print("=" * 60)
print("STEP 10: MULTICOLLINEARITY AUDIT — CATEGORICAL FEATURES (Cramér's V)")
print("=" * 60)

cat_audit_cols = [
    'DwellingTypeDescr', 'VentilationMethod', 'StructureType',
    'SuspendedWoodenFloor', 'CHBoilerThermostatControlled',
    'OBBoilerThermostatControlled', 'OBPumpInsideDwelling',
    'WarmAirHeatingSystem', 'UndergroundHeating',
    'CylinderStat', 'CombinedCylinder', 'ThermalMassCategory'
]
# Keep only columns that actually exist in the dataset
cat_audit_cols = [c for c in cat_audit_cols if c in df_raw.columns]
print(f"Categorical columns audited: {cat_audit_cols}\n")

def cramers_v(x, y):
    ct = pd.crosstab(x.astype(str), y.astype(str))
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    r, k = ct.shape
    phi2 = chi2 / n
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    denom = min(rc - 1, kc - 1)
    return np.sqrt(phi2c / denom) if denom > 0 else 0.0

# Sample for speed
rng = np.random.default_rng(42)
samp_idx = rng.choice(len(df_raw), size=min(100_000, len(df_raw)), replace=False)
df_cat_samp = df_raw[cat_audit_cols].iloc[samp_idx].reset_index(drop=True)

n = len(cat_audit_cols)
cv_matrix = pd.DataFrame(np.zeros((n, n)), index=cat_audit_cols, columns=cat_audit_cols)

print("Computing Cramér's V matrix ...")
for i, c1 in enumerate(cat_audit_cols):
    for j, c2 in enumerate(cat_audit_cols):
        if i == j:
            cv_matrix.loc[c1, c2] = 1.0
        elif j > i:
            v = cramers_v(df_cat_samp[c1], df_cat_samp[c2])
            cv_matrix.loc[c1, c2] = v
            cv_matrix.loc[c2, c1] = v

# Flag strongly associated pairs
print("\n--- Cramér's V: Strongly Associated Pairs (V > 0.5) ---")
strong_pairs = []
for i in range(n):
    for j in range(i+1, n):
        v = cv_matrix.iloc[i, j]
        if v > 0.5:
            strong_pairs.append((cat_audit_cols[i], cat_audit_cols[j], round(v, 4)))
if strong_pairs:
    for a, b, v in sorted(strong_pairs, key=lambda x: -x[2]):
        print(f"  {a:35s} <-> {b:35s}  V={v:.4f}")
else:
    print("  None found.")

# Full matrix
print("\n--- Full Cramér's V Matrix ---")
print(cv_matrix.round(3).to_string())

# Heatmap
mask_cv = np.triu(np.ones((n, n), dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cv_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            mask=np.zeros_like(cv_matrix, dtype=bool),  # show all
            annot_kws={'size': 8})
ax.set_title("Cramér's V — Categorical Feature Associations", fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('cramers_v_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: cramers_v_heatmap.png")


STEP 10: MULTICOLLINEARITY AUDIT — CATEGORICAL FEATURES (Cramér's V)
Categorical columns audited: ['DwellingTypeDescr', 'VentilationMethod', 'StructureType', 'SuspendedWoodenFloor', 'CHBoilerThermostatControlled', 'OBBoilerThermostatControlled', 'OBPumpInsideDwelling', 'WarmAirHeatingSystem', 'UndergroundHeating', 'CylinderStat', 'CombinedCylinder', 'ThermalMassCategory']

Computing Cramér's V matrix ...

--- Cramér's V: Strongly Associated Pairs (V > 0.5) ---
  CylinderStat                        <-> CombinedCylinder                     V=0.8448

--- Full Cramér's V Matrix ---
                              DwellingTypeDescr  VentilationMethod  StructureType  SuspendedWoodenFloor  CHBoilerThermostatControlled  OBBoilerThermostatControlled  OBPumpInsideDwelling  WarmAirHeatingSystem  UndergroundHeating  CylinderStat  CombinedCylinder  ThermalMassCategory
DwellingTypeDescr                         1.000              0.102          0.086                 0.096                         0.155 

---
## Step 11 — Mixed Multicollinearity (Categorical vs Continuous)

Calculate eta-squared (η²) from one-way ANOVA for each categorical × continuous pair. Flag pairs with η² > 0.14 (large effect size per Cohen). This identifies which categorical features are redundant with continuous ones.

In [20]:
# ── Step 11: Eta-squared — categorical vs continuous ─────────────────────────
from scipy.stats import f_oneway

print("=" * 60)
print("STEP 11: MIXED MULTICOLLINEARITY — ETA-SQUARED (η²)")
print("=" * 60)

def eta_squared(cat_series, cont_series):
    groups = [cont_series[cat_series == v].dropna().values
              for v in cat_series.unique() if len(cont_series[cat_series == v]) > 1]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return np.nan
    grand_mean = cont_series.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total   = ((cont_series - grand_mean) ** 2).sum()
    return ss_between / ss_total if ss_total > 0 else 0.0

# Use same sample
df_mixed = df_raw[cat_audit_cols + cont_cols].iloc[samp_idx].reset_index(drop=True)
# Convert categoricals to string for grouping
for c in cat_audit_cols:
    df_mixed[c] = df_mixed[c].astype(str)

print("Computing η² matrix (categorical × continuous) ...")
eta_matrix = pd.DataFrame(index=cat_audit_cols, columns=cont_cols, dtype=float)

for cat in cat_audit_cols:
    for cont in cont_cols:
        eta_matrix.loc[cat, cont] = eta_squared(df_mixed[cat], df_mixed[cont])

eta_matrix = eta_matrix.astype(float)

# Flag large effects
print("\n--- Categorical × Continuous Pairs with η² > 0.14 (Large Effect) ---")
large_effect = []
for cat in cat_audit_cols:
    for cont in cont_cols:
        v = eta_matrix.loc[cat, cont]
        if not np.isnan(v) and v > 0.14:
            large_effect.append((cat, cont, round(v, 4)))
if large_effect:
    large_effect.sort(key=lambda x: -x[2])
    print(f"  {'Categorical':<35} {'Continuous':<30} η²")
    print("  " + "-" * 75)
    for cat, cont, v in large_effect:
        print(f"  {cat:<35} {cont:<30} {v:.4f}")
else:
    print("  None found.")

# Heatmap
fig, ax = plt.subplots(figsize=(max(12, len(cont_cols) * 0.7), max(6, len(cat_audit_cols) * 0.6)))
sns.heatmap(eta_matrix, annot=True, fmt='.2f', cmap='Blues',
            vmin=0, vmax=1, linewidths=0.4, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Eta-Squared (η²): Categorical vs Continuous Features\n(η² > 0.14 = large effect)', fontsize=12, pad=10)
ax.set_xlabel('Continuous Features', fontsize=10)
ax.set_ylabel('Categorical Features', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('eta_squared_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: eta_squared_heatmap.png")
print("\n--- Full Eta-Squared Table ---")
print(eta_matrix.round(3).to_string())


STEP 11: MIXED MULTICOLLINEARITY — ETA-SQUARED (η²)
Computing η² matrix (categorical × continuous) ...

--- Categorical × Continuous Pairs with η² > 0.14 (Large Effect) ---
  Categorical                         Continuous                     η²
  ---------------------------------------------------------------------------
  DwellingTypeDescr                   FloorArea                      0.5788
  DwellingTypeDescr                   RoofArea                       0.5548
  DwellingTypeDescr                   WallArea                       0.4984
  DwellingTypeDescr                   LivingAreaPercent              0.4468
  DwellingTypeDescr                   NoStoreys                      0.3378
  VentilationMethod                   WHMainSystemEff                0.3296
  VentilationMethod                   HSMainSystemEfficiency         0.2734
  StructureType                       ThermalBridgingFactor          0.2715
  VentilationMethod                   UValueWindow                   

---
## Step 12 — Resolve `CylinderStat` and `CombinedCylinder`

NZV analysis flagged these for dropping, but FAMD places them as strong drivers on Factors 2 & 3. This targeted analysis checks:
1. Cramér's V with `DwellingTypeDescr`
2. Eta-squared with `BerRating`
3. Logistic regression accuracy predicting `DwellingTypeDescr` with vs without them
→ Final **Keep / Drop** recommendation.

In [21]:
# ── Step 12: CylinderStat & CombinedCylinder targeted analysis ───────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("STEP 12: RESOLVE CylinderStat & CombinedCylinder")
print("=" * 60)

target_cols = ['CylinderStat', 'CombinedCylinder']
target_cols = [c for c in target_cols if c in df_raw.columns]
print(f"Columns under analysis: {target_cols}")

# 1. Cramér's V with DwellingTypeDescr
print("\n1) Cramér's V with DwellingTypeDescr:")
for col in target_cols:
    v = cramers_v(df_raw[col].astype(str), df_raw['DwellingTypeDescr'].astype(str))
    print(f"   {col:<25} V = {v:.4f}  {'→ STRONGLY associated' if v > 0.5 else ''}")

# 2. Eta-squared with BerRating
print("\n2) Eta-squared (η²) with BerRating:")
for col in target_cols:
    eta = eta_squared(df_raw[col].astype(str), df_raw['BerRating'].dropna())
    print(f"   {col:<25} η² = {eta:.4f}  {'→ LARGE EFFECT' if eta > 0.14 else ('→ medium' if eta > 0.06 else '→ small')}")

# 3. Logistic Regression: predict DwellingTypeDescr
#    Build feature set: one-hot encode all categoricals + scale numerics
print("\n3) Logistic Regression: DwellingTypeDescr prediction with/without CylinderStat & CombinedCylinder")
samp_n = 30_000
rng2   = np.random.default_rng(0)
sidx   = rng2.choice(len(df_raw), size=min(samp_n, len(df_raw)), replace=False)
df_lr  = df_raw.iloc[sidx].copy().reset_index(drop=True)

# Encode target
le   = LabelEncoder()
y_lr = le.fit_transform(df_lr['DwellingTypeDescr'].astype(str))

base_feats = [c for c in df_raw.columns if c not in ['BerRating', 'DwellingTypeDescr', 'CountyName'] + target_cols]
full_feats = base_feats + target_cols

def build_X(df, feature_cols):
    num_c = [c for c in feature_cols if df[c].dtype in ['float32', 'float64', 'int32', 'int64']]
    cat_c = [c for c in feature_cols if c not in num_c]
    parts = []
    if num_c:
        parts.append(df[num_c].fillna(0).astype(float))
    if cat_c:
        parts.append(pd.get_dummies(df[cat_c].astype(str), drop_first=True))
    return pd.concat(parts, axis=1).fillna(0) if parts else pd.DataFrame()

pipe = make_pipeline(StandardScaler(with_mean=False), LogisticRegression(max_iter=300, C=0.5, random_state=42))

print("   Building feature matrices ...")
X_base = build_X(df_lr, base_feats)
X_full = build_X(df_lr, full_feats)

cv_base = cross_val_score(pipe, X_base, y_lr, cv=3, scoring='accuracy', n_jobs=-1)
cv_full = cross_val_score(pipe, X_full, y_lr, cv=3, scoring='accuracy', n_jobs=-1)

print(f"\n   Accuracy WITHOUT CylinderStat/CombinedCylinder: {cv_base.mean():.4f} ± {cv_base.std():.4f}")
print(f"   Accuracy WITH    CylinderStat/CombinedCylinder: {cv_full.mean():.4f} ± {cv_full.std():.4f}")
delta = cv_full.mean() - cv_base.mean()
print(f"   Delta: {delta:+.4f}")

print("\n--- FINAL RECOMMENDATION ---")
for col in target_cols:
    eta  = eta_squared(df_raw[col].astype(str), df_raw['BerRating'].dropna())
    cv_v = cramers_v(df_raw[col].astype(str), df_raw['DwellingTypeDescr'].astype(str))
    if eta > 0.06 or cv_v > 0.3 or delta > 0.005:
        verdict = "KEEP"
        reason  = f"η²={eta:.3f} vs BerRating, V={cv_v:.3f} with DwellingType, LR delta={delta:+.4f}"
    else:
        verdict = "DROP"
        reason  = f"Low η²={eta:.3f}, Low V={cv_v:.3f}, negligible LR gain ({delta:+.4f})"
    print(f"   {col:<25} → {verdict}  [{reason}]")


STEP 12: RESOLVE CylinderStat & CombinedCylinder
Columns under analysis: ['CylinderStat', 'CombinedCylinder']

1) Cramér's V with DwellingTypeDescr:
   CylinderStat              V = 0.1118  
   CombinedCylinder          V = 0.1249  

2) Eta-squared (η²) with BerRating:
   CylinderStat              η² = 0.0104  → small
   CombinedCylinder          η² = 0.0088  → small

3) Logistic Regression: DwellingTypeDescr prediction with/without CylinderStat & CombinedCylinder
   Building feature matrices ...


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_ite


   Accuracy WITHOUT CylinderStat/CombinedCylinder: 0.7609 ± 0.0025
   Accuracy WITH    CylinderStat/CombinedCylinder: 0.7595 ± 0.0023
   Delta: -0.0014

--- FINAL RECOMMENDATION ---
   CylinderStat              → DROP  [Low η²=0.010, Low V=0.112, negligible LR gain (-0.0014)]
   CombinedCylinder          → DROP  [Low η²=0.009, Low V=0.125, negligible LR gain (-0.0014)]


---
## Step 13 — Validate Removal Candidates

For each removal candidate — `WallArea`, `RoofArea`, `UValueWall`, `WindowArea`, `UValueFloor` — report their VIF, top-3 correlations, and correlation with `BerRating`. Determine whether they are truly unimportant or merely collinear (information captured elsewhere), and identify the **keeper** in each collinear group.

In [22]:
# ── Step 13: Validate removal candidates ─────────────────────────────────────
print("=" * 60)
print("STEP 13: VALIDATE REMOVAL CANDIDATES")
print("=" * 60)

removal_candidates = ['WallArea', 'RoofArea', 'UValueWall', 'WindowArea', 'UValueFloor']
removal_candidates = [c for c in removal_candidates if c in df_raw.columns]

# Correlation with BerRating (full dataset, no sample needed)
ber_corr = {}
for col in cont_cols + ['BerRating']:
    pass  # already have X_vif from Step 9

# Use full dataset correlations
df_num = df_raw[cont_cols + ['BerRating']].dropna().astype(float)
full_corr_matrix = df_num.corr()

# VIF lookup from Step 9
vif_lookup = dict(zip(vif_data['Feature'], vif_data['VIF']))

print(f"\n{'Feature':<18} {'VIF':>8}  {'r w/ BerRating':>16}  Top-3 Correlations")
print("-" * 100)
keeper_decisions = {}

for col in removal_candidates:
    vif_score = vif_lookup.get(col, np.nan)
    r_ber     = full_corr_matrix.loc[col, 'BerRating'] if 'BerRating' in full_corr_matrix.columns else np.nan

    # Top 3 correlations with other features (exclude BerRating and self)
    corr_row = full_corr_matrix[col].drop(labels=[col, 'BerRating'], errors='ignore')
    top3     = corr_row.abs().nlargest(3)
    top3_str = ', '.join([f"{feat}({val:.2f})" for feat, val in top3.items()])

    print(f"  {col:<16} {vif_score:>8.1f}  {r_ber:>16.4f}  {top3_str}")

    # Determine keeper within correlated group
    group_members = top3.index[top3 > 0.7].tolist() if not top3.empty else []
    if group_members:
        candidates_in_group = [col] + group_members
        # Keeper = highest |r| with BerRating
        r_vals = {c: abs(full_corr_matrix.loc[c, 'BerRating']) if c in full_corr_matrix.index else 0
                  for c in candidates_in_group if c in full_corr_matrix.index}
        if r_vals:
            keeper = max(r_vals, key=r_vals.get)
            keeper_decisions[col] = {'group': candidates_in_group, 'keeper': keeper,
                                     'keeper_r': r_vals.get(keeper, 0), 'candidate_r': abs(r_ber)}

print("\n--- Collinearity Group Analysis ---")
for col, info in keeper_decisions.items():
    verdict = "KEEP" if info['keeper'] == col else "DROP (collinear)"
    print(f"  {col:<16}: Group={info['group']}")
    print(f"               Keeper={info['keeper']} (|r|={info['keeper_r']:.4f} vs BerRating)")
    print(f"               {col} |r|={info['candidate_r']:.4f}  → {verdict}")
    print()

# For candidates NOT in a high-corr group
lone_candidates = [c for c in removal_candidates if c not in keeper_decisions]
if lone_candidates:
    print("\nCandidates with no strong correlates (|r|>0.7):")
    for col in lone_candidates:
        r_ber = full_corr_matrix.loc[col, 'BerRating']
        vif_s = vif_lookup.get(col, np.nan)
        verdict = "KEEP (unique info)" if abs(r_ber) > 0.1 or vif_s < 5 else "DROP (low VIF + low r)"
        print(f"  {col:<16} r={r_ber:.4f}, VIF={vif_s:.1f}  → {verdict}")


STEP 13: VALIDATE REMOVAL CANDIDATES

Feature                 VIF    r w/ BerRating  Top-3 Correlations
----------------------------------------------------------------------------------------------------
  WallArea              3.2           -0.0814  FloorArea(0.71), RoofArea(0.70), WindowArea(0.66)
  RoofArea              4.8           -0.0524  FloorArea(0.87), WallArea(0.70), WindowArea(0.64)
  UValueWall            2.4            0.6569  Year_of_Construction(0.65), UValueRoof(0.61), UValueWindow(0.53)
  WindowArea            2.2           -0.1920  WallArea(0.66), FloorArea(0.64), RoofArea(0.64)
  UValueFloor           2.4            0.5556  Year_of_Construction(0.58), UValueWall(0.51), UValueWindow(0.51)

--- Collinearity Group Analysis ---
  WallArea        : Group=['WallArea', 'FloorArea', 'RoofArea']
               Keeper=WallArea (|r|=0.0814 vs BerRating)
               WallArea |r|=0.0814  → KEEP

  RoofArea        : Group=['RoofArea', 'FloorArea', 'WallArea']
               K

---
## Step 14 — Final Consolidated Summary Report

Full feature table with VIF, correlation with BerRating, collinearity group, and Keep/Drop/Replace recommendation. Groups collinear features into named clusters and outputs the **final recommended feature set**.

In [23]:
# ── Step 14: Consolidated Summary Report ─────────────────────────────────────
import io

print("=" * 60)
print("STEP 14: FINAL CONSOLIDATED SUMMARY REPORT")
print("=" * 60)

# Define collinearity clusters (informed by VIF + correlation analysis above)
clusters = {
    'Size Cluster':       ['WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea'],
    'Insulation Cluster': ['UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor'],
    'Efficiency Cluster': ['HSMainSystemEfficiency', 'WHMainSystemEff', 'HSSupplSystemEff',
                           'HSEffAdjFactor', 'WHEffAdjFactor'],
    'Heat Fraction':      ['HSSupplHeatFraction', 'SHRenewableResources', 'WHRenewableResources',
                           'DistributionLosses'],
    'Ventilation Cluster':['NoOfFansAndVents', 'PercentageDraughtStripped', 'ThermalBridgingFactor'],
    'Building Cluster':   ['NoStoreys', 'NoOfSidesSheltered', 'LivingAreaPercent', 'Year_of_Construction'],
    'Heating Control':    ['NoCentralHeatingPumps', 'HeatSystemControlCat', 'HeatSystemResponseCat'],
    'Singletons':         ['SupplWHFuel', 'SupplSHFuel'],
}

# Build cluster lookup
feature_to_cluster = {}
for cluster_name, feats in clusters.items():
    for f in feats:
        feature_to_cluster[f] = cluster_name

# Determine recommendation for each continuous feature
def get_recommendation(col, vif_score, r_ber):
    if pd.isna(vif_score):
        return 'Keep', 'Categorical — handled separately'
    if vif_score > 10:
        # Check if it's the best in its cluster
        cluster = feature_to_cluster.get(col, 'Singletons')
        cluster_feats = [f for f in clusters.get(cluster, [col]) if f in full_corr_matrix.index]
        r_vals = {f: abs(full_corr_matrix.loc[f, 'BerRating']) for f in cluster_feats}
        keeper = max(r_vals, key=r_vals.get) if r_vals else col
        if keeper == col:
            return 'Keep', f'Highest |r| in {cluster} despite high VIF'
        else:
            return 'Drop', f'Collinear with {keeper} (VIF={vif_score:.1f}); {keeper} has higher |r|={r_vals.get(keeper,0):.3f}'
    elif vif_score > 5:
        if abs(r_ber) > 0.2:
            return 'Keep', f'Moderate VIF but strong signal |r|={abs(r_ber):.3f}'
        else:
            return 'Monitor', f'Moderate VIF, weak BerRating signal'
    else:
        return 'Keep', 'Low VIF, independent feature'

rows = []
for col in cont_cols:
    vif_s = vif_lookup.get(col, np.nan)
    r_ber = full_corr_matrix.loc[col, 'BerRating'] if col in full_corr_matrix.index else np.nan
    cluster = feature_to_cluster.get(col, '—')
    rec, reason = get_recommendation(col, vif_s, r_ber if not pd.isna(r_ber) else 0)
    rows.append({'Feature': col, 'VIF': round(vif_s, 1) if not pd.isna(vif_s) else '—',
                 'r(BerRating)': round(r_ber, 4) if not pd.isna(r_ber) else '—',
                 'Cluster': cluster, 'Recommendation': rec, 'Reason': reason})

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(['Recommendation', 'VIF'], ascending=[True, False])

print("\n--- Feature Summary Table ---")
print(f"{'Feature':<30} {'VIF':>8}  {'r(BER)':>8}  {'Cluster':<22} {'Rec':<10}  Reason")
print("-" * 120)
for _, row in summary_df.iterrows():
    print(f"  {row['Feature']:<28} {str(row['VIF']):>8}  {str(row['r(BerRating)']):>8}  "
          f"{row['Cluster']:<22} {row['Recommendation']:<10}  {row['Reason']}")

print("\n--- Collinearity Clusters ---")
for cname, feats in clusters.items():
    present = [f for f in feats if f in cont_cols]
    if present:
        r_vals  = {f: abs(full_corr_matrix.loc[f, 'BerRating']) if f in full_corr_matrix.index else 0 for f in present}
        keeper  = max(r_vals, key=r_vals.get)
        print(f"  {cname}: {present}")
        print(f"    → Keeper: {keeper}  (|r(BER)|={r_vals[keeper]:.4f})")

keep_set   = summary_df[summary_df['Recommendation'] == 'Keep']['Feature'].tolist()
drop_set   = summary_df[summary_df['Recommendation'] == 'Drop']['Feature'].tolist()
monit_set  = summary_df[summary_df['Recommendation'] == 'Monitor']['Feature'].tolist()

print(f"\n=== FINAL RECOMMENDED CONTINUOUS FEATURE SET ===")
print(f"  KEEP    ({len(keep_set):2d}): {keep_set}")
print(f"  MONITOR ({len(monit_set):2d}): {monit_set}")
print(f"  DROP    ({len(drop_set):2d}): {drop_set}")

# Save as text
buf = io.StringIO()
buf.write("=== MULTICOLLINEARITY AUDIT SUMMARY ===\n\n")
buf.write("--- Collinearity Clusters ---\n")
for cname, feats in clusters.items():
    present = [f for f in feats if f in cont_cols]
    if present:
        r_vals = {f: abs(full_corr_matrix.loc[f, 'BerRating']) if f in full_corr_matrix.index else 0 for f in present}
        keeper = max(r_vals, key=r_vals.get)
        buf.write(f"  {cname}: {present} -> Keeper: {keeper}\n")

buf.write("\n--- Feature Recommendations ---\n")
for _, row in summary_df.iterrows():
    buf.write(f"  {row['Feature']:<30} VIF={str(row['VIF']):>7}  r={str(row['r(BerRating)']):>8}  "
              f"[{row['Recommendation']}] {row['Reason']}\n")

buf.write(f"\nFINAL KEEP SET ({len(keep_set)} continuous features):\n")
for f in keep_set:
    buf.write(f"  {f}\n")
buf.write(f"\nMONITOR SET ({len(monit_set)} features):\n")
for f in monit_set:
    buf.write(f"  {f}\n")
buf.write(f"\nDROP SET ({len(drop_set)} features):\n")
for f in drop_set:
    buf.write(f"  {f}\n")

with open('multicollinearity_summary.txt', 'w') as f:
    f.write(buf.getvalue())
print("\nSaved: multicollinearity_summary.txt")

# Append to pca_summary_output.txt
with open('pca_summary_output.txt', 'a') as f:
    f.write("\n\n" + "=" * 60 + "\n")
    f.write("MULTICOLLINEARITY AUDIT (Steps 9-14)\n")
    f.write("=" * 60 + "\n")
    f.write(buf.getvalue())
print("Appended to: pca_summary_output.txt")

print("\nAll plots saved: vif_correlation_heatmap.png, cramers_v_heatmap.png, eta_squared_heatmap.png")


STEP 14: FINAL CONSOLIDATED SUMMARY REPORT

--- Feature Summary Table ---
Feature                             VIF    r(BER)  Cluster                Rec         Reason
------------------------------------------------------------------------------------------------------------------------
  SHRenewableResources             77.9    0.2364  Heat Fraction          Drop        Collinear with HSSupplHeatFraction (VIF=77.9); HSSupplHeatFraction has higher |r|=0.307
  WHRenewableResources             77.9     0.237  Heat Fraction          Drop        Collinear with HSSupplHeatFraction (VIF=77.9); HSSupplHeatFraction has higher |r|=0.307
  HSEffAdjFactor                   26.2    0.0344  Efficiency Cluster     Drop        Collinear with WHMainSystemEff (VIF=26.2); WHMainSystemEff has higher |r|=0.436
  WHEffAdjFactor                   25.9    0.0308  Efficiency Cluster     Drop        Collinear with WHMainSystemEff (VIF=25.9); WHMainSystemEff has higher |r|=0.436
  WHMainSystemEff               